In [2]:
import sys
import pandas as pd
from pathlib import Path
sys.path.append(str(Path.cwd().parent / 'src'))
from pipeline import get_retriever, answer

R = get_retriever()
ROOT = Path.cwd().parent
gold = pd.read_csv(ROOT / 'data' / 'golden' / 'golden_set.csv', dtype=str)

results = []
print("Running Evaluation...\n")

for _, row in gold.iterrows():
    q = row['question']
    is_in_scope = row['in_scope'] == '1'
    
    out = answer(q, R)
    
    # Check retrieval metrics (only for in-scope)
    if is_in_scope:
        gold_secs = set(row['gold_sections'].split(';'))
        # FIX: Cast retrieved sections to strings so they match the CSV
        retrieved_secs = [str(R.by_id[i]['section']) for i in out['ids']]
        hit = any(s in gold_secs for s in retrieved_secs)
        
        print(f"Q: {q}")
        print(f"  Refused: {out['abstained']} | Hit Gold Section: {hit}")
        print(f"  Answer: {out['answer'][:100]}...\n")
    else:
        print(f"Q: {q}")
        print(f"  Refused correctly: {out['abstained']}\n")
        
    results.append(out)

Running Evaluation...

Q: what are the employer duties regarding protective equipment?
  Refused: False | Hit Gold Section: True
  Answer: Employers have several duties regarding protective equipment:

1. They must provide adequate and sui...

Q: who appoints safety representatives?
  Refused: False | Hit Gold Section: True
  Answer: The Minister shall make regulations to provide for the appointment of safety representatives in pres...

Q: what are the rules for fencing machinery?
  Refused: False | Hit Gold Section: True
  Answer: The rules for fencing machinery include the requirement that every flywheel and moving part of a pri...

Q: what is the penalty for obstructing an inspector?
  Refused: False | Hit Gold Section: False
  Answer: The penalty for obstructing an inspector is a fine not exceeding forty-eight currency points, or a t...

Q: How do I register a company?
  Refused correctly: True

Q: What is the income tax rate in Uganda?
  Refused correctly: True

